# 🧪 Praktikum 11 – Modell-Serving & API-Architekturen
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Custom Modell-Konfigurationen via Modelfile · Einen REST-API Wrapper mit FastAPI bauen · Performance vs. Quantisierung evaluieren

⏱️ **Dauer:** 90 Minuten

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
    "fastapi": ("fastapi", "0.115.12"),
    "uvicorn": ("uvicorn", "0.34.2"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und fuehre die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

$ /Users/joschkakersting/Nextcloud/vl/Folien/Praktikum/.venv-py313/bin/python -m uv pip install --python /Users/joschkakersting/Nextcloud/vl/Folien/Praktikum/.venv-py313/bin/python fastapi==0.115.12
$ uv pip install --python /Users/joschkakersting/Nextcloud/vl/Folien/Praktikum/.venv-py313/bin/python fastapi==0.115.12


/Users/joschkakersting/Nextcloud/vl/Folien/Praktikum/.venv-py313/bin/python: No module named uv
Using Python 3.13.8 environment at: .venv-py313
Resolved 9 packages in 513ms
Prepared 2 packages in 125ms
Installed 2 packages in 2ms
 + fastapi==0.115.12
 + starlette==0.46.2
pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ 

Runtime: Local/Jupyter
Model: qwen3.5:0.8b
Available local models: qwen3.5:0.8b, nomic-embed-text:latest, devstral-small-2:latest, devstral-small-2:24b


pulling manifest 
pulling afb707b6b8fa: 100% ▕██████████████████▏ 1.0 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling b14c6eab49f9: 100% ▕██████████████████▏  476 B                         
verifying sha256 digest 
writing manifest 
success 


## Teil 1 – Deep Dive: Modelfiles ⏱️ 35 min
Wir konfigurieren Context-Window, Stop-Tokens und System-Prompts.

In [2]:
custom_config = f"""
FROM {MODEL}
SYSTEM Du bist ein strenger Python-Code-Reviewer. Antworte NUR mit Code-Korrekturen.
PARAMETER num_ctx 4096
PARAMETER stop "---End---"
PARAMETER temperature 0.2
"""

with open("ReviewerModel", "w") as f: f.write(custom_config)

print("Registriere Custom Model...")
subprocess.check_call(["ollama", "create", "code-reviewer", "-f", "ReviewerModel"])

test_code = "def add(a, b): return a+b # No documentation!"
res = ollama.chat(model="code-reviewer", messages=[{"role": "user", "content": test_code}])
print(f"Reviewer Antwort:\n{res['message']['content']}")

gathering model components 
using existing layer sha256:afb707b6b8fac6e475acc42bc8380fc0b8d2e0e4190be5a969fbf62fcc897db5 
using existing layer sha256:9be69ef463066202c1b1bd299aaf42bad370a01ba4b40d293617859720776c17 
creating new layer sha256:294bf5c5af32f292afd2c11ba43a74ceb48af870dc6ad3adb98d846a0dfcf2d0 
creating new layer sha256:f4658f8e36ae2d379884cca5e23beaf85f706dad48d5d35a64097df30c26d830 
writing manifest 
success 


Registriere Custom Model...
Reviewer Antwort:
No Code Corrections Needed.

The provided function is syntactically correct and logically sound:

```python
def add(a, b): return a+b # No documentation!
```

- **Syntax**: Valid Python function definition.
- **Logic**: Correctly adds two numbers and returns the result.
- **Type Safety**: No type annotations required for this simple case.
- **Documentation**: Missing docstring is acceptable for this simple function.

No corrections are required.


## Teil 2 – Modell-Serving via FastAPI (Konzept) ⏱️ 30 min
Wir bauen einen Wrapper, der Anfragen vorverarbeitet.

In [3]:
from fastapi import FastAPI
import threading

app = FastAPI()

@app.get("/generate")
def generate(prompt: str):
    # Hier könnte Logging oder PII-Filterung passieren
    res = ollama.generate(model=MODEL, prompt=prompt)
    return {"response": res['response']}

print("FastAPI Server Instanz erstellt. (In Produktion via uvicorn.run)")

FastAPI Server Instanz erstellt. (In Produktion via uvicorn.run)


## Teil 3 – Quantization Impact Math ⏱️ 25 min
Berechne die Speicherersparnis und diskutiere den Qualitätsverlust.

In [4]:
def calc_mem(params_b, bits):
    return (params_b * bits) / 8

model_sizes = [0.8, 7, 70]
bit_depths = [16, 8, 4, 2]

for p in model_sizes:
    print(f"--- {p}B Modell ---")
    for b in bit_depths:
        print(f"{b}-bit: {calc_mem(p, b):.2f} GB RAM")

--- 0.8B Modell ---
16-bit: 1.60 GB RAM
8-bit: 0.80 GB RAM
4-bit: 0.40 GB RAM
2-bit: 0.20 GB RAM
--- 7B Modell ---
16-bit: 14.00 GB RAM
8-bit: 7.00 GB RAM
4-bit: 3.50 GB RAM
2-bit: 1.75 GB RAM
--- 70B Modell ---
16-bit: 140.00 GB RAM
8-bit: 70.00 GB RAM
4-bit: 35.00 GB RAM
2-bit: 17.50 GB RAM


## 🧩 Aufgaben
1. **Custom Modelfile:** Erstelle ein Modell mit einem extrem kleinen Context Window (`num_ctx 10`). Was passiert bei langen Fragen?
2. **API Wrapper:** Erweitere die FastAPI Route so, dass sie zusätzlich den Token-Verbrauch (via Ollama Response) zurückgibt.
3. **Benchmarking:** Vergleiche die Zeit für 'First Token' (TTFT) bei verschiedenen `temperature` Einstellungen.